# UCI Online Retail II — EU macro & weather pipeline

Mirrors **`Notebooks/data_prep.ipynb`** for **UCI Online Retail II** (EU countries only), using **UK/EU macro** (Eurostat + FRED) and **country-proxy weather** (Open-Meteo) instead of US FRED + US city weather.

| Step | What | Output |
|------|------|--------|
| 1 | Download & clean UCI Online Retail II (EU filter) | ~799k invoice lines |
| 2a | EU macro (HICP, unemployment, interest) | `eu_macro_by_country.csv` |
| 2b | Weather by country proxy city (1 API call / country) | `eu_weather_by_country.csv` |
| 2c | Join calendar + macro + weather | enriched fact table |
| 3 | Synthetic loyalty CRM | tier / points / sensitivity |
| 4 | Roll up to `uci_dim_customers` | ~5.8k customers |
| 5 | Save Parquet + feature store + uplift/NBA | `data/modeling/uci_*.parquet` |

**Weather fetch:** set `CACHE_ONLY = False` in the setup cell (default) to download from Open-Meteo on first run (~4 min). Set `True` to skip API calls when the CSV cache already exists.

**CLI:** `python scripts/uci_pipeline.py` runs the full pipeline (same as Step 5).

Helpers: `scripts/uci_context.py` · Loader: `load_online_retail_ii(..., eu_only=True)` in `scripts/modeling_extensions.py`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)


def find_project_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / "scripts" / "build_datasets.py").exists():
            return candidate
    return path


PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_MODELING = PROJECT_ROOT / "data" / "modeling"
DATA_EXTERNAL = DATA_RAW / "external"

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import importlib
import modeling_extensions
import uci_context

importlib.reload(modeling_extensions)
importlib.reload(uci_context)

from modeling_extensions import load_online_retail_ii
from uci_context import (
    COUNTRY_META,
    OpenMeteoQuotaExceeded,
    add_context_uci,
    build_uci_dim_customers,
    country_to_geo,
    generate_synthetic_crm,
    load_eu_macro,
    load_eu_weather,
    order_weather_keys,
)

MACRO_PATH = DATA_EXTERNAL / "eu_macro_by_country.csv"
WEATHER_PATH = DATA_EXTERNAL / "eu_weather_by_country.csv"

# False = fetch missing weather from Open-Meteo (~21 API calls, ~4 min first run).
# True  = use eu_weather_by_country.csv only (fast rebuild when cache exists).
CACHE_ONLY = False

## Step 1 — Download & clean UCI Online Retail II (EU only)

**Source:** UCI ML Repository (`online_retail_II.xlsx`, two sheets)  
**Grain:** one row = one **invoice line** (product on an order)  
**Cleaning:** drop null `customer_id`, returns (`quantity <= 0`), invalid prices  
**Geography:** keep only countries in `COUNTRY_META` (Eurostat + weather proxy) — drops ~0.9% of rows outside EU coverage

In [2]:
retail = load_online_retail_ii(DATA_RAW, eu_only=True)

CONTEXT_DATE_START = retail["order_date"].min().normalize()
CONTEXT_DATE_END = retail["order_date"].max().normalize()
CONTEXT_MONTH_START = CONTEXT_DATE_START.strftime("%Y-%m")
CONTEXT_MONTH_END = CONTEXT_DATE_END.strftime("%Y-%m")
MACRO_FETCH_MONTH_START = (
    (CONTEXT_DATE_START - pd.DateOffset(months=12)).strftime("%Y-%m")
)

print(f"Rows: {len(retail):,}")
print(f"Customers: {retail['customer_id'].nunique():,}")
print(f"Orders: {retail['order_id'].nunique():,}")
print(f"Date range: {CONTEXT_DATE_START.date()} → {CONTEXT_DATE_END.date()}")
print(f"\nTop countries:")
print(retail["country"].value_counts().head(10))
retail.head(3)

Rows: 798,626
Customers: 5,798
Orders: 36,684
Date range: 2009-12-01 → 2011-12-09

Top countries:
country
United Kingdom    725250
Germany            16694
EIRE               15743
France             13812
Netherlands         5088
Spain               3719
Belgium             3068
Switzerland         3011
Portugal            2446
Italy               1468
Name: count, dtype: int64


,order_id,product_id,product_name,quantity,order_date,unit_price,customer_id,country,line_total
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0


## Step 2a — Download EU macro (HICP, unemployment, interest)

**Sources:**
- **CPI (HICP):** Eurostat `PRC_HICP_MIDX` (all-items `CP00`)
- **Unemployment:** Eurostat `UNE_RT_M` (seasonally adjusted, total)
- **Interest rate:** FRED OECD immediate rates — UK gilt yield, ECB euro-area rate for eurozone members, plus country series for Denmark, Norway, Sweden, Poland, Czech Republic, and Switzerland

**Grain:** one row = **month × country** (via Eurostat `geo` code)  
**Join later:** `month` + `geo` (mapped from transaction `country`)

In [11]:
countries_in_data = retail["country"].dropna().unique()
geos_needed = sorted({country_to_geo(c) for c in countries_in_data if country_to_geo(c)})
print(f"Countries with Eurostat mapping: {len(geos_needed)} geo codes")
print(geos_needed)

macro = load_eu_macro(
    geos_needed,
    MACRO_FETCH_MONTH_START,
    CONTEXT_MONTH_END,
    cache_path=MACRO_PATH,
)

print(f"\nMacro rows: {len(macro):,} | saved → {MACRO_PATH.relative_to(PROJECT_ROOT)}")
print(f"Month range: {macro['month'].min()} → {macro['month'].max()}")
macro.head(8)

Countries with Eurostat mapping: 21 geo codes
['AT', 'BE', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EL', 'ES', 'FI', 'FR', 'IE', 'IT', 'LT', 'MT', 'NL', 'NO', 'PL', 'PT', 'SE', 'UK']

Macro rows: 777 | saved → data\raw\external\eu_macro_by_country.csv
Month range: 2008-12 → 2011-12


,month,country,geo,cpi_index,inflation_mom,inflation_yoy,unemployment_rate,interest_rate
0,2008-12,Austria,AT,88.42,NaN,NaN,6.700000,2.4864
1,2009-01,Austria,AT,87.90,-0.00588,NaN,6.666667,1.8122
2,2009-02,Austria,AT,88.33,0.00489,NaN,6.033333,1.2571
3,2009-03,Austria,AT,88.49,0.00181,NaN,6.933333,1.0620
4,2009-04,Austria,AT,88.65,0.00181,NaN,7.433333,0.8419
5,2009-05,Austria,AT,88.71,0.00068,NaN,7.166667,0.7821
6,2009-06,Austria,AT,88.62,-0.00101,NaN,7.666667,0.6980
7,2009-07,Austria,AT,88.26,-0.00406,NaN,8.166667,0.3576


## Step 2b — Download weather by country proxy

**Source:** Open-Meteo geocoding + archive API  
**Grain:** one row = **date + country** (proxy capital city per country)  
**Strategy:** **one API call per country** over the full order-date range (min → max). Completes in a single run (~21 calls, ~4 min). Progress is saved after each country to `eu_weather_by_country.csv`.

Online Retail has no city field — we use one representative location per country (e.g. London for UK, Berlin for Germany). The cache stores all calendar days in each country's range; the join in Step 2c keeps only dates with orders.

**`CACHE_ONLY`** (set in setup cell):
- `False` (default) — download any missing weather from Open-Meteo
- `True` — skip API calls; use existing CSV only

In [4]:
order_keys = order_weather_keys(
    retail,
    str(CONTEXT_DATE_START.date()),
    str(CONTEXT_DATE_END.date()),
)
print(f"Need weather for {len(order_keys):,} order-date+country rows")
print(f"Across {order_keys['country'].nunique()} countries with proxy mapping")
print(f"CACHE_ONLY = {CACHE_ONLY}")

try:
    weather = load_eu_weather(order_keys, WEATHER_PATH, cache_only=CACHE_ONLY, verbose=True)
except OpenMeteoQuotaExceeded as exc:
    print(f"Weather paused: {exc}")
    print("Wait a few minutes and re-run — completed countries are already saved.")
    weather = pd.read_csv(WEATHER_PATH, parse_dates=["date"]) if WEATHER_PATH.exists() else pd.DataFrame()

if not weather.empty or WEATHER_PATH.exists():
    cache = pd.read_csv(WEATHER_PATH, parse_dates=["date"]) if WEATHER_PATH.exists() else weather
    cache_keys = cache[["date", "country"]].drop_duplicates()
    coverage = order_keys.merge(cache_keys, on=["date", "country"], how="inner")
    missing = len(order_keys) - len(coverage)
    pct = len(coverage) / len(order_keys) if len(order_keys) else 0
    print(
        f"\nWeather cache: {len(cache):,} rows | {pct:.1%} of order keys covered"
    )
    print(
        f"Order keys: {order_keys['date'].min().date()} -> {order_keys['date'].max().date()}"
    )
    if missing:
        print(f"Still missing: {missing:,} keys — set CACHE_ONLY=False and re-run")
    else:
        print("All order-date+country keys have weather.")
else:
    print("No weather cache yet — set CACHE_ONLY=False in the setup cell and run this cell.")

weather.head(3) if not weather.empty else cache.head(3)

Need weather for 2,590 order-date+country rows
Across 21 countries with proxy mapping
Weather cache already complete for all order keys.

Weather: 14,152 cache rows | 100.0% of order keys covered
Order keys: 2009-12-01 → 2011-12-09
All order-date+country keys have weather.


,date,temp_c,rain_mm,snow_cm,precip_mm,weather_code,wind_gust_kmh,had_rain,had_snow,had_major_storm,country
0,2009-12-01,2.9,0.3,0.0,0.3,51,37.8,True,False,False,United Kingdom
1,2009-12-02,7.8,3.5,0.0,3.5,53,42.1,True,False,False,United Kingdom
2,2009-12-03,5.5,1.1,0.0,1.1,53,50.4,True,False,False,United Kingdom


## Step 2c — Join context onto transactions

1. **Calendar features** — day-of-week, month start/end, EU retail events (Black Friday week, Christmas season, back-to-school, country-specific sales)
2. **Country holidays** — `is_public_holiday` per country (GB, DE, FR, …)
3. **EU macro** — `LEFT JOIN` on `month` + `geo`
4. **Weather** — `LEFT JOIN` on `date` + `country` (100% coverage when Step 2b cache is complete)

In [12]:
df = add_context_uci(retail, macro, weather)

context_cols = [
    "order_date", "country", "geo", "month",
    "is_public_holiday", "is_retail_spending_day",
    "is_black_friday_week", "is_cyber_monday_week", "is_christmas_season",
    "is_back_to_school", "is_major_sale_period",
    "days_to_christmas", "days_to_black_friday",
    "is_sinterklaas", "is_three_kings_day", "is_french_winter_sale",
    "cpi_index", "inflation_mom", "unemployment_rate", "interest_rate",
    "temp_c", "rain_mm", "had_major_storm",
]
present = [c for c in context_cols if c in df.columns]
print("Enriched shape:", df.shape)
print("\nNull rates on context columns:")
for col in present:
    if col not in ("order_date", "country", "geo", "month"):
        print(f"  {col:24s} {df[col].isna().mean():.1%}")

sale_flags = [c for c in present if c.startswith("is_")]
print("\nSale-period hit rates:")
for col in sale_flags:
    print(f"  {col:24s} {df[col].mean():.2%}")

df[present].head(3)

Enriched shape: (798626, 47)

Null rates on context columns:
  is_public_holiday        0.0%
  is_retail_spending_day   0.0%
  is_black_friday_week     0.0%
  is_cyber_monday_week     0.0%
  is_christmas_season      0.0%
  is_back_to_school        0.0%
  is_major_sale_period     0.0%
  days_to_christmas        0.0%
  days_to_black_friday     0.0%
  is_sinterklaas           0.0%
  is_three_kings_day       0.0%
  is_french_winter_sale    0.0%
  cpi_index                0.0%
  inflation_mom            0.0%
  unemployment_rate        0.0%
  interest_rate            0.0%
  temp_c                   0.0%
  rain_mm                  0.0%
  had_major_storm          0.0%

Sale-period hit rates:
  is_public_holiday        0.23%
  is_retail_spending_day   1.28%
  is_black_friday_week     3.04%
  is_cyber_monday_week     4.43%
  is_christmas_season      19.73%
  is_back_to_school        7.63%
  is_major_sale_period     27.62%
  is_sinterklaas           0.00%
  is_three_kings_day       0.00%
  is_fre

,order_date,country,geo,month,is_public_holiday,is_retail_spending_day,is_black_friday_week,is_cyber_monday_week,is_christmas_season,is_back_to_school,is_major_sale_period,days_to_christmas,days_to_black_friday,is_sinterklaas,is_three_kings_day,is_french_winter_sale,cpi_index,inflation_mom,unemployment_rate,interest_rate,temp_c,rain_mm,had_major_storm
0,2009-12-01 07:45:00,United Kingdom,UK,2009-12,False,False,False,True,True,False,True,23,359,False,False,False,88.0,0.00571,10.966667,3.8871,2.9,0.3,False
1,2009-12-01 07:45:00,United Kingdom,UK,2009-12,False,False,False,True,True,False,True,23,359,False,False,False,88.0,0.00571,10.966667,3.8871,2.9,0.3,False
2,2009-12-01 07:45:00,United Kingdom,UK,2009-12,False,False,False,True,True,False,True,23,359,False,False,False,88.0,0.00571,10.966667,3.8871,2.9,0.3,False


## Step 3 — Join synthetic loyalty CRM

Same tier/points simulation as Superstore `data_prep.ipynb` — one row per `customer_id`, joined onto every line.

In [14]:
customer_ids = df["customer_id"].drop_duplicates().sort_values()
crm = generate_synthetic_crm(customer_ids, seed=42)
print("CRM lookup:", crm.shape)
crm.head(3)

CRM lookup: (5798, 6)


,customer_id,tier,points_balance,email_opt_in,app_usage_score,discount_sensitivity
0,12346,Gold,4618,False,0.84,0.12
1,12348,Platinum,8072,True,0.80,0.20
2,12349,Silver,1750,False,0.51,0.69


Create a transaction-level fact table by generating unique transaction IDs and enriching transactional data with CRM and external features (customer attributes, macro conditions, and environmental signals) via a left join on customer_id.

In [15]:
df = df.reset_index(drop=True)
df["transaction_id"] = df.index.map(lambda i: f"uci_{i:07d}")

fact = df.merge(crm, on="customer_id", how="left")
print("uci_fact_transactions shape:", fact.shape)
fact[[
    "transaction_id", "customer_id", "country", "line_total",
    "tier", "points_balance", "cpi_index", "temp_c",
]].head(3)

uci_fact_transactions shape: (798626, 53)


,transaction_id,customer_id,country,line_total,tier,points_balance,cpi_index,temp_c
0,uci_0000000,13085,United Kingdom,83.4,Gold,3562,88.0,2.9
1,uci_0000001,13085,United Kingdom,81.0,Gold,3562,88.0,2.9
2,uci_0000002,13085,United Kingdom,81.0,Gold,3562,88.0,2.9


## Step 4 — Roll up to `uci_dim_customers`

**Grain change:** line items → one row per customer (lifetime revenue, recency, CRM attributes).

In [16]:
dim = build_uci_dim_customers(fact)
print("uci_dim_customers shape:", dim.shape)
dim.head(5)

uci_dim_customers shape: (5798, 16)


,customer_id,country,first_order_date,last_order_date,total_orders,total_lines,total_revenue,avg_unit_price,tier,points_balance,email_opt_in,app_usage_score,discount_sensitivity,avg_order_value,recency_days,tenure_days
0,12346,United Kingdom,2009-12-14 08:34:00,2011-01-18 10:01:00,12,34,77556.46,6.100000,Gold,4618,False,0.84,0.12,6463.038333,325,400
1,12348,Finland,2010-09-27 14:59:00,2011-09-25 13:13:00,5,51,2019.40,3.786275,Platinum,8072,True,0.80,0.20,403.880000,74,362
2,12349,Italy,2010-04-29 13:20:00,2011-11-21 09:51:00,4,175,4428.69,8.459657,Silver,1750,False,0.51,0.69,1107.172500,18,570
3,12350,Norway,2011-02-02 16:01:00,2011-02-02 16:01:00,1,17,334.40,3.841176,Bronze,433,True,0.36,0.63,334.400000,309,0
4,12352,Norway,2010-11-12 10:20:00,2011-11-03 14:37:00,10,103,2849.84,13.676796,Gold,3107,True,0.60,0.50,284.984000,35,356


## Step 5 — Save modeling Parquet files

Writes all `uci_*.parquet` files to `data/modeling/` (separate from the Superstore spine).

Uses `scripts/uci_pipeline.py` — respects **`CACHE_ONLY`** from the setup cell for weather. Equivalent CLI:

```bash
python scripts/uci_pipeline.py
```

In [17]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from uci_pipeline import build_uci_spine

outputs = build_uci_spine(cache_only_weather=CACHE_ONLY)

for name, path in outputs.items():
    n = len(pd.read_parquet(path))
    print(f"{name:28s} {n:>12,} rows  ->  {path.relative_to(PROJECT_ROOT)}")

Weather cache already complete for all order keys.
uci_fact_transactions             798,626 rows  ->  data\modeling\uci_fact_transactions.parquet
uci_dim_customers                   5,798 rows  ->  data\modeling\uci_dim_customers.parquet
uci_customer_features               5,798 rows  ->  data\modeling\uci_customer_features.parquet
uci_uplift_campaigns               11,596 rows  ->  data\modeling\uci_uplift_campaigns.parquet
uci_nba_offer_catalog                   8 rows  ->  data\modeling\uci_nba_offer_catalog.parquet
uci_nba_offer_events                5,798 rows  ->  data\modeling\uci_nba_offer_events.parquet


## Join summary

```
UCI Online Retail (line items)
  + calendar + country holidays     (computed in-place)
  + EU retail event flags           (Black Friday week, Christmas season, …)
  + EU macro                        LEFT JOIN on month + geo
  + country-proxy weather           LEFT JOIN on date + country
  + synthetic CRM                   LEFT JOIN on customer_id
  = uci_fact_transactions

uci_fact_transactions  ->  GROUP BY customer_id  ->  uci_dim_customers
```

**Country proxy map** (weather + Eurostat geo): see `COUNTRY_META` in `scripts/uci_context.py`.  
Step 1 filters to these countries only. With a complete weather cache (`CACHE_ONLY=False` on first run), macro and weather joins have no nulls on retained rows.